In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.cluster import KMeans
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import binarize
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


#Getting the needed datasets
pokemon_data = pd.read_csv("/kaggle/input/datasets/andrejkaz/pokemon-card-masterset/pokemon-tcg-data-master 1999-2023.csv")
price_data = pd.read_csv("/kaggle/input/datasets/andrejkaz/pokemon-price-dataset/pokemon_pricecharting.csv")

In [ ]:
#===Preparing the data===#

#Columns of interest for the pokemon data set
coi_pokemon_data = ["id", "set", "series", "generation", "name", "release_date", "hp", "attacks", "rarity"]

#Pokemon column data
pkcd = pokemon_data[coi_pokemon_data].copy()
pkcd.head()

#Columns of interest for the price data set
coi_price_data = ["used_price", "graded_price", "prod_name", "set_name"]

#Price column data
pccd = price_data[coi_price_data].copy()
pccd.head()

#Sort by pokemon id
pkcd_sorted = pkcd.sort_values("id", ascending = True)

#Sort by price
pccd_sorted = pccd.sort_values(["used_price", "graded_price"])
#Rename the column names
pccd_sorted.rename(columns = {"prod_name" : "name", "set_name" : "set"}, inplace = True)

#Defining with regex expressions 
def clean_column(ds):
    return (
        ds
        .str.replace("pokemon-", "", regex=False)
        .str.replace("-", " ", regex=False)
        .str.replace("_", " ", regex=False)
        .str.replace("%27", "'", regex=False)
        .str.replace("#", " ", regex=False)  
        .str.replace(r'\s*\d+[a-z]*$', '', regex=True) 
        .str.replace(r'(gx|ex|vmax|vstar|v)\b', r' \1', regex=True) 
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .str.lower()
    )

#Fix the set mappings
pccd_sorted["set"] = clean_column(pccd_sorted["set"])
pkcd_sorted["set"] = clean_column(pkcd_sorted["set"])

#Fix the name mappings
pccd_sorted["name"] = clean_column(pccd_sorted["name"])
pkcd_sorted["name"] = clean_column(pkcd_sorted["name"])

#Merge the 2 datasets
merged_df = pd.merge(pccd_sorted, pkcd_sorted, on=["set", "name"], how="inner")

#Missing values = Support card (no atk, no hp) and some cards have 0.00 price
merged_df.isnull().sum()

#Drop the rows with 0.00 (Only if both used_price and graded_price are 0.00) 
merged_df["used_price"] = merged_df["used_price"].replace(0, np.nan)
merged_df["graded_price"] = merged_df["graded_price"].replace(0, np.nan)
merged_df = merged_df.dropna(subset = ["used_price", "graded_price"])
merged_df = merged_df.drop(columns=["attacks"])
merged_df = merged_df.dropna(subset=["hp"])
merged_df["release_date"] = pd.to_datetime(merged_df["release_date"])
merged_df["year_date"] = merged_df["release_date"].dt.year


#Missing val hp an attacks for support cards
merged_df.isnull().sum()


Preparing the data for the model, merging needed data sets as dropping un-needed columns and checking for any missing values. 

In [ ]:
#Improving the data for training 
#Converting categorical val to numerical that (kinda) make sense for training the model
key_store = []
rarity_ = []

for index, row in merged_df.iterrows():
    float(index)
    key_store.append([index, row["rarity"]])

rarity_ = [i[1] for i in key_store]
rarity_level = []

#Try to convert data 
def rarity_lvl(status):
    match status:
        case 'Common':
            return 1.00
        case 'Uncommon':
            return 2.00
        case 'Rare':
            return 3.00
        case 'Rare Holo':
            return 4.00
        case 'Amazing Rare':
            return 5.00
        case 'Rare Holo GX':
            return 6.00
        case 'Rare Holo V':
            return 7.00
        case 'Rare Holo VMAX':
            return 8.00
        case 'Rare Holo VSTAR':
            return 9.00
        case 'Rare Ultra':
            return 10.00
        case 'Rare Rainbow':
            return 11.00
        case 'Rare Shining':
            return 12.00
        case 'Radiant Rare':
            return 13.00
        case 'Double Rare':
            return 14.00
        case 'Ultra Rare':
            return 15.00
        case 'Illustration Rare':
            return 16.00
        case 'Special Illustration Rare':
            return 17.00
        case 'Rare Secret':
            return 18.00
        case 'Hyper Rare':
            return 19.00
        case _:
            return None

def gen_num(status):
    match status:
        case 'Seventh':
            return 1
        case 'Eighth':
            return 2
        case 'Ninth':
            return 3
        case _:
            return None


merged_df["rarity_level"] = merged_df["rarity"].apply(rarity_lvl)
merged_df["gen_num"] = merged_df["generation"].apply(gen_num)

merged_df.head(45000)
merged_df.isnull().sum()

#Cleaning up the data
df = merged_df.copy()
col_to_drop = ["set", "id", "series", "generation", "release_date", "rarity"]

df.drop(columns = col_to_drop, inplace = True)
df.info()
df.describe()
df.isnull().sum()
df.head(45000)

Cleaning up the data before training the model

In [ ]:
#===Making clusters before training the data (To see card relations)===#
km = KMeans(init = "random", n_clusters = 6)
km.fit(merged_df[["used_price", "graded_price"]])

#Get the labels of the clusters
km.labels_
#Get the centers
km.cluster_centers_

#Style
colors = ["red", "orange", "navy", "cyan", "magenta", "yellow"]
cluster_colors = [colors[label] for label in km.labels_]

#Represent the clustered data
plt.scatter(
    merged_df["used_price"],
    merged_df["graded_price"],
    c = cluster_colors,
    edgecolors = "white",
    linewidths = 0.5,
    s = 30
)

#Represent the clusters
plt.scatter(
    km.cluster_centers_[:, 0],
    km.cluster_centers_[:, 1],
    c = "black",
    marker = "X",
    s = 80,
)

#Show the clusters
plt.xlabel("Used Price")
plt.ylabel("Graded Price")
plt.show()

Making cluster just to see relations between the values

In [ ]:
#===Making the model===#

#Training data
train_df = df.copy()
train_df["used_x_rarity"] = train_df["used_price"] * train_df["rarity_level"]
train_df["hp_x_year"] = train_df["hp"] * train_df["year_date"]
train_df["gen_num"] = binarize(train_df[["gen_num"]])

#Improving needed training data
x = train_df[["used_price", "hp", "year_date", "gen_num", "rarity_level", "used_x_rarity", "hp_x_year"]].copy()
x["used_price"] = np.log1p(x["used_price"])
x["used_x_rarity"] = np.log1p(x["used_x_rarity"])
x["hp_x_year"] = np.log1p(x["hp_x_year"])
y = np.log1p(train_df["graded_price"])

x_train, x_test, y_train, y_test = train_test_split(x, y, train_size = 0.70, random_state = 42)

#Standard scaler
sc = StandardScaler()
x_train_s = sc.fit_transform(x_train)
x_test_s = sc.transform(x_test)

#Linear regression
lr = LinearRegression()
lr.fit(x_train_s, y_train)
y_pred_lr_log = lr.predict(x_test_s)

#Model
model = Sequential([
    #Input layer
    Dense(32, input_dim = x_train_s.shape[1], activation = "relu",kernel_initializer = "he_normal"),
    BatchNormalization(),
    Dropout(0.1),

    #Hidden layers
    Dense(64, kernel_initializer = "he_normal", activation = "relu"),
    BatchNormalization(),
    Dropout(0.15),
    
    Dense(128, kernel_initializer = "he_normal", activation = "relu"),
    BatchNormalization(),
    Dropout(0.15),

    Dense(64, kernel_initializer = "he_normal", activation = "relu"),
    BatchNormalization(),
    Dropout(0.15),
    
    Dense(32, kernel_initializer = "he_normal", activation = "relu"),
    BatchNormalization(),
    Dropout(0.15),

    #Output layer
    Dense(1)
])

#Compile the model
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = "huber", metrics = ['mae'])

#Callback, and save the best version of the model
checkpoint = ModelCheckpoint("best_model.keras", save_best_only = True, monitor = "val_loss", mode = "min", verbose = 1)

history = model.fit(
    x_train_s, y_train,
    validation_split = 0.25,
    batch_size = 32,
    epochs = 100,
    verbose = 1,
    callbacks = [checkpoint]
)

#Evaluation
y_pred_lr  = np.expm1(y_pred_lr_log)
y_pred_nn  = np.expm1(model.predict(x_test_s).flatten())
y_test_actual = np.expm1(y_test)

mae_lr  = mean_absolute_error(y_test_actual, y_pred_lr)
r2_lr   = r2_score(y_test_actual, y_pred_lr)

mae_nn  = mean_absolute_error(y_test_actual, y_pred_nn)
rmse_nn = np.sqrt(mean_squared_error(y_test_actual, y_pred_nn))
r2_nn   = r2_score(y_test_actual, y_pred_nn)

print(f"Linear Regression MAE: {mae_lr:.4f}, r^2: {r2_lr:.4f}")
print(f"Neural Network MAE: {mae_nn:.4f}, rmse: {rmse_nn:.4f}, r^2: {r2_nn:.4f}")

#Bins and their labels
bins = [0, 50, 150, 500, np.inf]

labels = ["<$50", "$50-$150", "$150-$500", "$500+"]

y_test_binned = pd.cut(y_test_actual, bins = bins, labels = labels)
y_pred_binned = pd.cut(pd.Series(y_pred_nn), bins = bins, labels = labels)

#Results and summary
accuracy = accuracy_score(y_test_binned, y_pred_binned)
print(f"Bin Accuracy: {accuracy:.1%}")

model.summary()
model.save("my_model.keras")

Making the model as well as using linear regression model for the baseline. Training the model and getting the results as well as the needed evaluation

In [ ]:
#Comparison of results
results = pd.DataFrame({"Actual": y_test_actual.values,"Predicted": y_pred_nn,})
results["Difference"] = results["Actual"] - results["Predicted"]

#First 20
print(results.head(20))

#Last 20
print(results.sort_values("Actual", ascending=False).head(20))


In [ ]:
#Visual representation
plt.figure(figsize = (8, 8))
plt.scatter(results["Actual"], results["Predicted"], alpha = 0.2, color = "cyan", edgecolors = "blue")

#Prediction line
min_val = min(results["Actual"].min(), results['Predicted'].min())
max_val = max(results["Actual"].max(), results['Predicted'].max())
plt.plot([min_val, max_val], [min_val, max_val],"r--", linewidth = 1.5)

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.show()